# RMS state scales under the observation distribution

This notebook estimates a fixed scale for each family and state coordinate:

\[
S_{X,k}=\sqrt{\mathbb E_{\theta\sim\Pi,\;t\sim U[0,3.2]}[X_{\theta,k}(t)^2]}.
\]

Parameters and times are independently sampled. This is **uncentered RMS**, not standard deviation, maximum amplitude, or the average of trajectory-specific RMS values. We integrate over the **observation** interval, not the query interval. No observation noise is added.

The repository's solution function takes normalized time \(s=t/4\), so we sample \(s\sim U[0,0.8]\). Each draw consists of a fresh parameter vector and one fresh time. All coordinates use the repository's existing independent uniform parameter priors. For Stuart–Landau, separate scales are calculated for x and y.

This is a calibration calculation only. It does **not** change any training scales, checkpoints, or configuration. No analytical check is used.

In [1]:
from pathlib import Path
import sys, json, hashlib, csv, time
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "ode_paper").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "ode_paper").is_dir():
    raise RuntimeError("Open from Paper repo or its notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))
from ode_paper.families import SPECS, bounds, solution

CALIBRATION_SEED = 20261019  # Separate from all training/validation/test data.
REPLICATES = 8
SAMPLES_PER_REPLICATE = 2**20
CHUNK_SIZE = 65536
NORMALIZED_OBSERVATION_END = 0.8
PHYSICAL_TIME_DIVISOR = 4.0
Q = 0.02
FAMILIES = tuple(SPECS)
print("Samples per family:", f"{REPLICATES * SAMPLES_PER_REPLICATE:,}")
print("Physical observation interval: [0, 3.2]; normalized interval: [0, 0.8]")
print("Calibration seed:", CALIBRATION_SEED)
print("No training settings will be changed.")


Samples per family: 8,388,608
Physical observation interval: [0, 3.2]; normalized interval: [0, 0.8]
Calibration seed: 20261019
No training settings will be changed.


## Sampling and uncertainty

Each family and replicate has separate random streams for parameters and time. Sampling proceeds in chunks to limit memory, but every sample is a new independent joint draw. We check support and empirical uniform moments for every parameter and time coordinate, and check empirical parameter/time correlations.

For \(Z_k=X_k^2\), we estimate \(\widehat S_k=\sqrt{\overline Z_k}\). The displayed standard error uses the Monte Carlo sample variance and the delta method:
\[
\operatorname{SE}(\widehat S_k)\approx\frac{\sqrt{\widehat{\operatorname{Var}}(Z_k)/N}}{2\widehat S_k}.
\]
The intervals are approximate 95% Monte Carlo intervals for the calibration calculation, not uncertainty intervals for trained-model performance. Convergence rows reuse cumulative samples and are therefore correlated.

In [2]:
def calibrate_family(family, family_seed):
    lo, hi = bounds(family)
    p, m = len(lo), len(SPECS[family]["scale"])
    count = 0
    sum_z, sum_z2 = np.zeros(m), np.zeros(m)
    sum_u, sum_u2 = np.zeros(p + 1), np.zeros(p + 1)
    sum_uu = np.zeros((p + 1, p + 1))
    minimum, maximum = np.full(p + 1, np.inf), np.full(p + 1, -np.inf)
    replicate_rms, convergence = [], []
    started = time.perf_counter()
    for replicate, rep_seed in enumerate(family_seed.spawn(REPLICATES), 1):
        theta_seed, time_seed = rep_seed.spawn(2)
        theta_rng, time_rng = np.random.default_rng(theta_seed), np.random.default_rng(time_seed)
        rep_sum_z = np.zeros(m)
        for start in range(0, SAMPLES_PER_REPLICATE, CHUNK_SIZE):
            size = min(CHUNK_SIZE, SAMPLES_PER_REPLICATE - start)
            theta = theta_rng.uniform(lo, hi, size=(size, p))
            normalized_time = time_rng.uniform(0, NORMALIZED_OBSERVATION_END, size=(size, 1))
            assert np.all(theta >= lo) and np.all(theta <= hi)
            assert np.all(normalized_time >= 0) and np.all(normalized_time < 0.8)
            physical_states = solution(theta, normalized_time, family)[:, 0, :]
            assert physical_states.shape == (size, m)
            if not np.isfinite(physical_states).all():
                raise FloatingPointError(f"Nonfinite clean trajectory in {family}")
            z = physical_states**2
            sum_z += z.sum(axis=0)
            sum_z2 += (z**2).sum(axis=0)
            rep_sum_z += z.sum(axis=0)
            # Transform independent uniforms to [0,1] for common sampling diagnostics.
            u = np.column_stack(((theta - lo) / (hi - lo), normalized_time[:, 0] / 0.8))
            sum_u += u.sum(axis=0)
            sum_u2 += (u**2).sum(axis=0)
            sum_uu += u.T @ u
            minimum = np.minimum(minimum, u.min(axis=0))
            maximum = np.maximum(maximum, u.max(axis=0))
            count += size
        replicate_rms.append(np.sqrt(rep_sum_z / SAMPLES_PER_REPLICATE).tolist())
        if replicate in (1, 2, 4, 8) or replicate == REPLICATES:
            convergence.append({"samples": count, "rms": np.sqrt(sum_z / count).tolist()})
        print(f"{family}: replicate {replicate}/{REPLICATES}, RMS={np.sqrt(sum_z / count)}", flush=True)
    mean_square = sum_z / count
    rms = np.sqrt(mean_square)
    variance_z = (sum_z2 - count * mean_square**2) / (count - 1)
    assert np.all(variance_z >= 0)
    se = np.sqrt(variance_z / count) / (2 * rms)
    u_mean = sum_u / count
    u_variance = (sum_u2 - count * u_mean**2) / (count - 1)
    covariance = (sum_uu - count * np.outer(u_mean, u_mean)) / (count - 1)
    correlation = covariance / np.sqrt(np.outer(u_variance, u_variance))
    np.fill_diagonal(correlation, 0)
    # Wide sanity thresholds (not hypothesis tests or a proof of independence).
    assert np.max(np.abs(u_mean - 0.5)) < 8 * np.sqrt(1 / (12 * count))
    assert np.max(np.abs(u_variance - 1 / 12)) < 0.001
    assert np.max(np.abs(correlation)) < 0.005
    return {
        "parameter_names": SPECS[family]["names"],
        "parameter_low": lo.tolist(), "parameter_high": hi.tolist(),
        "state_coordinates": ["x", "y"] if m == 2 else ["x"],
        "samples": count, "mean_square": mean_square.tolist(), "rms": rms.tolist(),
        "rms_standard_error": se.tolist(),
        "rms_approx_95pct_interval": np.column_stack((rms - 1.96 * se, rms + 1.96 * se)).tolist(),
        "physical_noise_std_if_adopted": (Q * rms).tolist(),
        "existing_state_scale": list(SPECS[family]["scale"]),
        "replicate_rms": replicate_rms, "convergence": convergence,
        "uniform_diagnostics": {
            "coordinate_order": SPECS[family]["names"] + ["time"],
            "means_expected_0.5": u_mean.tolist(),
            "variances_expected_1_over_12": u_variance.tolist(),
            "minimum": minimum.tolist(), "maximum": maximum.tolist(),
            "max_abs_offdiagonal_correlation": float(np.max(np.abs(correlation))),
        },
        "seconds": time.perf_counter() - started,
    }

family_seeds = np.random.SeedSequence(CALIBRATION_SEED).spawn(len(FAMILIES))
results = {family: calibrate_family(family, seed) for family, seed in zip(FAMILIES, family_seeds)}


forced_linear: replicate 1/8, RMS=[0.65318967]


forced_linear: replicate 2/8, RMS=[0.65326131]


forced_linear: replicate 3/8, RMS=[0.65317998]


forced_linear: replicate 4/8, RMS=[0.65336182]


forced_linear: replicate 5/8, RMS=[0.65320094]


forced_linear: replicate 6/8, RMS=[0.65313105]


... (22 outputs left out)

stuart_landau: replicate 5/8, RMS=[0.65654221 0.67974412]


stuart_landau: replicate 6/8, RMS=[0.65644999 0.67984654]


stuart_landau: replicate 7/8, RMS=[0.65641558 0.67982356]


stuart_landau: replicate 8/8, RMS=[0.65640671 0.67984844]


In [3]:
print(f"{'Family':18} {'Coord':5} {'RMS':>12} {'95% MC halfwidth':>18} {'q * RMS':>12} {'Old scale':>12}")
for family, result in results.items():
    for k, coordinate in enumerate(result["state_coordinates"]):
        print(f"{family:18} {coordinate:5} {result['rms'][k]:12.7f} "
              f"{1.96 * result['rms_standard_error'][k]:18.7f} "
              f"{result['physical_noise_std_if_adopted'][k]:12.7f} "
              f"{result['existing_state_scale'][k]:12.7f}")
print("\nThese are prospective RMS scales; existing training constants remain unchanged.")


Family             Coord          RMS   95% MC halfwidth      q * RMS    Old scale
forced_linear      x        0.6532414          0.0003552    0.0130648    0.8237455
logistic           x        0.7160788          0.0002187    0.0143216    1.6000000
birth_death        x        1.0066637          0.0006742    0.0201333    2.0000000
stuart_landau      x        0.6564067          0.0001768    0.0131281    1.0954451
stuart_landau      y        0.6798484          0.0001604    0.0135970    1.0954451

These are prospective RMS scales; existing training constants remain unchanged.


In [4]:
for family, result in results.items():
    print("\n" + family)
    for row in result["convergence"]:
        print(f"  N={row['samples']:>10,}: {np.array(row['rms'])}")
    diag = result["uniform_diagnostics"]
    print("  Uniform coordinate means:", np.round(diag["means_expected_0.5"], 6))
    print("  Maximum absolute cross-coordinate correlation:", diag["max_abs_offdiagonal_correlation"])



forced_linear
  N= 1,048,576: [0.65318967]
  N= 2,097,152: [0.65326131]
  N= 4,194,304: [0.65336182]
  N= 8,388,608: [0.65324143]
  Uniform coordinate means: [0.499948 0.500096 0.500056 0.499939]
  Maximum absolute cross-coordinate correlation: 0.00032613920258849614

logistic
  N= 1,048,576: [0.71531881]
  N= 2,097,152: [0.71569186]
  N= 4,194,304: [0.71591914]
  N= 8,388,608: [0.7160788]
  Uniform coordinate means: [0.500098 0.500139 0.499885 0.499946]
  Maximum absolute cross-coordinate correlation: 0.0004865239821540241

birth_death
  N= 1,048,576: [1.00640821]
  N= 2,097,152: [1.00647087]
  N= 4,194,304: [1.00684571]
  N= 8,388,608: [1.00666366]
  Uniform coordinate means: [0.499992 0.499845 0.499909 0.499983]
  Maximum absolute cross-coordinate correlation: 0.0008594530980795256

stuart_landau
  N= 1,048,576: [0.65665885 0.67977374]
  N= 2,097,152: [0.6562959  0.68002758]
  N= 4,194,304: [0.65654968 0.67972456]
  N= 8,388,608: [0.65640671 0.67984844]
  Uniform coordinate means: 

## Save the calibration record

The JSON includes prior ranges, sample counts, source fingerprint, uncertainty, and sampling diagnostics. The CSV is a compact coordinatewise summary. The existing scales in `src/ode_paper/families.py` are not edited.

In [5]:
OUTPUT = ROOT / "calibration"
OUTPUT.mkdir(exist_ok=True)
record = {
    "definition": "sqrt(E_theta~prior,E_t~Uniform(0,3.2)[X_theta,k(t)^2])",
    "method": "independent Monte Carlo joint parameter-time samples; no noise; no centering",
    "seed": CALIBRATION_SEED, "replicates": REPLICATES,
    "samples_per_replicate": SAMPLES_PER_REPLICATE, "chunk_size": CHUNK_SIZE,
    "physical_observation_interval": [0.0, 3.2], "normalized_observation_interval": [0.0, 0.8],
    "numpy_version": np.__version__,
    "families_source_sha256": hashlib.sha256((ROOT / "src/ode_paper/families.py").read_bytes()).hexdigest(),
    "training_scales_changed": False, "families": results,
}
(OUTPUT / "rms_observation_interval.json").write_text(json.dumps(record, indent=2, allow_nan=False))
with (OUTPUT / "rms_observation_interval.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["family", "coordinate", "samples", "rms", "standard_error", "ci95_low", "ci95_high",
                     "physical_noise_std_q002", "existing_state_scale"])
    for family, result in results.items():
        for k, coordinate in enumerate(result["state_coordinates"]):
            writer.writerow([family, coordinate, result["samples"], result["rms"][k], result["rms_standard_error"][k],
                             *result["rms_approx_95pct_interval"][k],
                             result["physical_noise_std_if_adopted"][k], result["existing_state_scale"][k]])
print("Saved:", OUTPUT / "rms_observation_interval.json")
print("Saved:", OUTPUT / "rms_observation_interval.csv")


Saved: <repo>/calibration/rms_observation_interval.json
Saved: <repo>/calibration/rms_observation_interval.csv
